# 🧠 SmolLM-135M — Baseline vs Fine-Tuned
## Comparative Evaluation & Continued Training Pipeline

> **A complete, runnable walkthrough:** fine-tune a small medical language model, measure the impact of one extra training epoch, and compare outputs side-by-side using the shared 60-question MediLearn evaluation set. The merged model produced here is saved to Google Drive and loaded directly by Modules 3, 4, and 6.

---

### 📋 Notebook sections

| # | Section | What happens |
|---|---------|--------------|
| 1 | **Setup** | Pin compatible library versions, restart kernel, verify GPU |
| 2 | **Background** | Why SLMs, LoRA, and this dataset |
| 3 | **Config** | All hyperparameters (identical to original run) |
| 4 | **Eval questions** | 10 medical Q&A pairs used throughout |
| 5 | **Baseline eval** | Run original SmolLM-135M-Instruct, no fine-tuning |
| 6 | **Fine-tuning** | Load 10-epoch adapter → train 1 more epoch |
| 7 | **Post-training eval** | Same 10 questions on the updated model |
| 8 | **Comparison** | Charts, tables, qualitative analysis |
| 9 | **Save & download** | JSON + Markdown report, ZIP download |

### 🔗 Resources
- **GitHub:** [mohres/LLM-SLM-Fine-tuning](https://github.com/mohres/LLM-SLM-Fine-tuning)
- **Fine-tuned model (10ep):** [mohres/SmolLM-135M-Instruct-medical_meadow_medical_flashcards-10epochs](https://huggingface.co/mohres/SmolLM-135M-Instruct-medical_meadow_medical_flashcards-10epochs)
- **Base model:** [HuggingFaceTB/SmolLM-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM-135M-Instruct)
- **Dataset:** [medalpaca/medical_meadow_medical_flashcards](https://huggingface.co/datasets/medalpaca/medical_meadow_medical_flashcards)

> ⚠️ **Before running:** set `Runtime → Change runtime type → T4 GPU`


---
## ⚙️ Section 1 — Environment Setup

### Why we pin versions
The `trl` / `transformers` / `peft` ecosystem moves fast and breaks often. The error:
```
ImportError: cannot import name 'is_liger_kernel_available' from 'transformers.utils'
```
happens when `trl ≥ 0.13` is paired with `transformers < 4.47`. We pin a **known-good set** that mirrors the versions used in the original training run.

| Package | Pinned version |
|---------|---------------|
| `transformers` | 4.47.1 |
| `peft` | 0.14.0 |
| `trl` | 0.13.0 |
| `datasets` | 2.21.0 |
| `accelerate` | 1.2.1 |
| `tokenizers` | 0.21.0 |

> 💡 After installation the cell calls `os.kill` to **restart the kernel automatically** so the fresh package versions take effect before any imports.


In [1]:
# ── Step 1a: Install pinned, compatible versions ──────────────────────────────
# Do NOT change versions unless you know what you're doing.
import subprocess, sys

pkgs = [
    "transformers==4.47.1",
    "tokenizers==0.21.0",
    "datasets==2.21.0",
    "peft==0.14.0",
    "trl==0.13.0",
    "accelerate==1.2.1",
    "bitsandbytes",
    "matplotlib",
    "pandas",
]

print("Installing packages (this takes ~2 min)…")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs,
    capture_output=True, text=True
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
else:
    print("✅ All packages installed successfully")
print("\n⚡ Restarting kernel to activate new versions…")


Installing packages (this takes ~2 min)…
STDERR: /Users/sudhanshusingh/Projects/DHS-SLM-Workshop/.venv/bin/python: No module named pip


⚡ Restarting kernel to activate new versions…


In [2]:
# ── Step 1b: Kernel restart (runs automatically after install cell in Colab) ──
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import os, signal
    print("⚡ Restarting kernel (Colab only) to activate new versions…")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("✅ Local environment: kernel restart not required.")


✅ Local environment: kernel restart not required.


> ✅ **After the kernel restarts**, continue from the next cell. The restart is expected — do not re-run the install cells.


In [3]:
# ── Step 1c: GPU check & core imports (run after kernel restart) ──────────────
import os, json, time, warnings
import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
from typing import List, Dict
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

# Check GPU availability (CUDA, MPS, or CPU)
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"\n🖥️  Device : {DEVICE.upper()}")

if DEVICE == "cuda":
    import subprocess
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if result.returncode == 0:
        print(result.stdout)
    else:
        print(f"🎮 GPU    : {torch.cuda.get_device_name(0)}")
        print(f"💾 VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
elif DEVICE == "mps":
    print("🎮 GPU    : Apple Silicon (MPS)")
else:
    print("⚠️  No GPU found. Some training/evaluation cells will be slow.")

# Verify versions
import transformers, peft, trl, datasets
print(f"\n📦 Package versions")
print(f"   transformers : {transformers.__version__}")
print(f"   peft         : {peft.__version__}")
print(f"   trl          : {trl.__version__}")
print(f"   datasets     : {datasets.__version__}")



🖥️  Device : MPS
🎮 GPU    : Apple Silicon (MPS)

📦 Package versions
   transformers : 5.14.1
   peft         : 0.20.0
   trl          : 1.9.2
   datasets     : 5.0.1


---
## 📁 Section 1d — Shared Workshop Folder (Google Drive)

Every module in this workshop reads from and writes to the same Google Drive folder. This is what
makes the case study continuous: the merged model this notebook produces is the exact model
Modules 3, 4, and 6 load — no module re-downloads or substitutes a different model.

In [4]:
import sys
import os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSHOP_DIR     = "/content/drive/MyDrive/SLM Workshop"
else:
    WORKSHOP_DIR     = os.path.abspath("../SLM_Workshop")

EVAL_DIR         = f"{WORKSHOP_DIR}/eval"
DRIVE_RESULTS_DIR = f"{EVAL_DIR}/results"
MERGED_MODEL_DIR = f"{WORKSHOP_DIR}/medical_slm_finetuned"   # Modules 3, 4, and 6 load from exactly this path

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"📁 Shared workshop folder: {WORKSHOP_DIR}")
print(f"   Eval set + results : {EVAL_DIR}")
print(f"   Merged model goes to: {MERGED_MODEL_DIR}  (created in Section 7.1 below)")


📁 Shared workshop folder: /Users/sudhanshusingh/Projects/DHS-SLM-Workshop/SLM_Workshop
   Eval set + results : /Users/sudhanshusingh/Projects/DHS-SLM-Workshop/SLM_Workshop/eval
   Merged model goes to: /Users/sudhanshusingh/Projects/DHS-SLM-Workshop/SLM_Workshop/medical_slm_finetuned  (created in Section 7.1 below)


---
## 🏛️ Section 2 — Background & Motivation

### Why fine-tune a Small Language Model?

| Property | SmolLM-135M (SLM) | GPT-4 (LLM) |
|----------|-------------------|-------------|
| Parameters | 135 M | ~1.8 T (est.) |
| Runs locally | ✅ | ❌ |
| Data stays on-premise | ✅ | ❌ |
| Cost to run | Free | Paid API |
| Fine-tunable on laptop/Colab | ✅ | ❌ |

### What is LoRA?

**LoRA (Low-Rank Adaptation)** freezes the original model and inserts tiny trainable "adapter" matrices at attention layers:

```
Base weights W  (frozen, 135M params)
      ↓
 + A · B         (LoRA adapters, ~200K params — 0.15% of total)
      ↓
Fine-tuned behaviour at <1% the training cost
```

This is why we can run epoch 11 in Colab in ~20 minutes.

### The dataset

[`medalpaca/medical_meadow_medical_flashcards`](https://huggingface.co/datasets/medalpaca/medical_meadow_medical_flashcards) — ~34,000 Q&A pairs derived from Anki medical curriculum cards covering:

| Domain | Examples |
|--------|---------|
| 🫀 Physiology | Bowman's space, cardiac output, blood pressure |
| 💊 Pharmacology | Drug mechanisms, antimicrobials, receptors |
| 🧫 Pathology | Cancer prognosis, disease markers |
| 🧠 Neuroscience | Cranial nerves, neurotransmitters |


---
## 🔧 Section 3 — Configuration

All hyperparameters are copied **verbatim** from the [HuggingFace model card](https://huggingface.co/mohres/SmolLM-135M-Instruct-medical_meadow_medical_flashcards-10epochs). Nothing is changed — we just run one more epoch on top.


In [5]:
# ── Model & dataset identifiers ───────────────────────────────────────────────
BASE_MODEL_ID = "HuggingFaceTB/SmolLM-135M-Instruct"
FINETUNED_ID  = "mohres/SmolLM-135M-Instruct-medical_meadow_medical_flashcards-10epochs"
DATASET_ID    = "medalpaca/medical_meadow_medical_flashcards"
OUTPUT_DIR    = "./smollm_epoch11"
RESULTS_DIR   = "./eval_results"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)

# ── Precision: bf16 on A100/H100, fp16 on T4/V100 ────────────────────────────
supports_bf16 = DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if supports_bf16 else torch.float16

# ── 4-bit NF4 quantization config (QLoRA) ─────────────────────────────────────
# Base weights are stored in 4-bit NormalFloat (NF4); compute happens in bf16/fp16.
# Double quantization additionally quantizes the quantization constants themselves.
from transformers import BitsAndBytesConfig

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = COMPUTE_DTYPE,
    bnb_4bit_use_double_quant = True,
)

# ── Training hyperparameters (from HF model card) ────────────────────────────
TRAIN_CONFIG = dict(
    learning_rate               = 1e-3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 2,        # effective batch = 32
    num_train_epochs            = 1,        # ONE additional epoch (epoch 11 on top of the published 10-epoch adapter)
    seed                        = 42,
    lr_scheduler_type           = "constant",
    warmup_ratio                = 0.05,
    optim                       = "adamw_torch",
    bf16                        = supports_bf16,
    fp16                        = DEVICE in ["cuda", "mps"] and not supports_bf16,
    logging_steps               = 10,
    save_strategy               = "epoch",
    eval_strategy               = "epoch",
    report_to                   = "none",
    output_dir                  = OUTPUT_DIR,
)

# ── LoRA config (from original training) ─────────────────────────────────────
LORA_R             = 16
LORA_ALPHA         = 32
LORA_DROPOUT       = 0.05
LORA_TARGET_MODS   = ["q_proj", "v_proj"]
MAX_SEQ_LEN        = 512

print("✅ Configuration set")
print(f"   Precision        : {'bf16' if supports_bf16 else 'fp16' if DEVICE=='cuda' else 'fp32'}")
print(f"   Quantization     : 4-bit NF4 (double quant) — base weights only")
print(f"   Effective batch  : {TRAIN_CONFIG['per_device_train_batch_size'] * TRAIN_CONFIG['gradient_accumulation_steps']}")
print(f"   Additional epochs: {TRAIN_CONFIG['num_train_epochs']}")
print(f"   LoRA rank / alpha: {LORA_R} / {LORA_ALPHA}")


✅ Configuration set
   Precision        : fp32
   Quantization     : 4-bit NF4 (double quant) — base weights only
   Effective batch  : 32
   Additional epochs: 1
   LoRA rank / alpha: 16 / 32


---
## ❓ Section 4 — Shared MediLearn Evaluation Set (60 questions)

This replaces the 10 ad hoc questions used in earlier drafts of this notebook. Every module now
scores against the **same fixed 60-question set** — 6 medical categories × 10 questions, in three
difficulty tiers (L1 basic recall, L2 clinical scenario, L3 advanced reasoning) — so a score here is
directly comparable to a score in Module 1, 3, 4, or 6.

> 📁 **Loaded from Drive, not hardcoded.** This notebook reads `medilearn_eval_60.json` directly from the shared workshop folder ([link](https://drive.google.com/drive/folders/1Vjr9sDkcE38zEu5kgFwGVnNM1bom7-Cr)) after mounting Drive — see Section 1d. Add it as a shortcut in your own Drive first; there is no embedded fallback copy.

In [6]:
import json

# ── Load the shared MediLearn eval set — no hardcoded/embedded fallback ──────
# The canonical file lives in the shared workshop Drive folder:
#   https://drive.google.com/drive/folders/1Vjr9sDkcE38zEu5kgFwGVnNM1bom7-Cr
# For it to be visible at EVAL_DIR after `drive.mount()`, add it to your own
# Drive as a shortcut ("Organize → Add shortcut" in Drive's UI) so it resolves
# under /content/drive/MyDrive/... — Colab cannot read someone else's Drive
# folder directly, only your own (which the shortcut makes it appear as).
#
# File format: {"name", "version", "description", "difficulty_legend", "questions": [...]}
# — a metadata wrapper around the 60-question list, not a bare list.

eval_set_path = f"{EVAL_DIR}/medilearn_eval_60.json"

if not os.path.exists(eval_set_path):
    raise FileNotFoundError(
        f"\n❌ Eval set not found at: {eval_set_path}\n\n"
        f"This notebook reads the shared 60-question MediLearn set directly from Drive —\n"
        f"it does not fall back to an embedded copy.\n\n"
        f"Fix: add the shared folder as a shortcut in your own Drive, then re-mount:\n"
        f"  https://drive.google.com/drive/folders/1Vjr9sDkcE38zEu5kgFwGVnNM1bom7-Cr\n"
        f"  (Drive UI → right-click the folder → 'Organize' → 'Add shortcut')\n"
        f"so it resolves at: {EVAL_DIR}/medilearn_eval_60.json"
    )

with open(eval_set_path) as f:
    _payload = json.load(f)

MEDILEARN_QUESTIONS = _payload["questions"] if isinstance(_payload, dict) else _payload
print(f"✅ Loaded shared eval set from Drive: {eval_set_path}")

assert len(MEDILEARN_QUESTIONS) == 60, f"Expected 60 questions, found {len(MEDILEARN_QUESTIONS)}"
from collections import Counter
diff_counts = Counter(q["difficulty"] for q in MEDILEARN_QUESTIONS)
cat_counts = Counter(q["category"] for q in MEDILEARN_QUESTIONS)
print(f"   L1: {diff_counts['L1']}  |  L2: {diff_counts['L2']}  |  L3: {diff_counts['L3']}  |  Total: {len(MEDILEARN_QUESTIONS)}")
print(f"   Categories: {dict(cat_counts)}")


✅ Loaded shared eval set from Drive: /Users/sudhanshusingh/Projects/DHS-SLM-Workshop/SLM_Workshop/eval/medilearn_eval_60.json
   L1: 18  |  L2: 24  |  L3: 18  |  Total: 60
   Categories: {'Cardiology': 10, 'Neurology': 10, 'Respiratory': 10, 'Endocrine': 10, 'Critical Care': 10, 'Pharmacology': 10}


---
## 🛠️ Section 5 — Helper Functions

The scoring harness below (`term_coverage_score`, `semantic_relevance_score`, `run_medilearn_eval`,
`summarize_medilearn`) is copy-identical to the one in Module 1, and will be copy-identical in
Modules 3, 4, and 6 too.


In [7]:
def generate_response(model, tokenizer, instruction: str, max_new_tokens: int = 128) -> str:
    """Inference with chat template — matches HF model card exactly."""
    messages = [{"role": "user", "content": instruction}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # Use the tokenizer's __call__ (not .encode) so we get a real attention_mask
    # back. SmolLM's pad_token == eos_token, so without an explicit mask the
    # model can't tell "real token" from "padding" and warns on every call.
    encoded = tokenizer(input_text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"],
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full.split("assistant\n", 1)[-1].strip() if "assistant\n" in full else full.strip()


# Shared scoring harness — copy-identical in Modules 1, 2, 3, 4, and 6.
# Same questions, same scoring code, same embedding model = directly comparable numbers.
!pip install -q sentence-transformers

import time
import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading embedding model for semantic-relevance scoring (BAAI/bge-small-en-v1.5)...")
_embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
_ref_embeddings = _embedder.encode(
    [q["reference_answer"] for q in MEDILEARN_QUESTIONS],
    normalize_embeddings=True, show_progress_bar=False
)

def term_coverage_score(response: str, key_terms: list) -> float:
    """Fraction of expected key terms present in the response (0-1)."""
    if not key_terms:
        return 0.0
    resp_lower = response.lower()
    hits = sum(1 for t in key_terms if t.lower() in resp_lower)
    return round(hits / len(key_terms), 3)

def semantic_relevance_score(response: str, ref_embedding) -> float:
    """Cosine similarity between the response and the reference answer (0-1)."""
    resp_emb = _embedder.encode([response], normalize_embeddings=True, show_progress_bar=False)[0]
    return round(float(np.dot(resp_emb, ref_embedding)), 3)

def run_medilearn_eval(generate_fn, model_label: str, questions=None, verbose=True):
    """generate_fn(question_text:str) -> response_text:str. Returns a results DataFrame."""
    qs = questions if questions is not None else MEDILEARN_QUESTIONS
    ref_embs = _ref_embeddings if questions is None else _embedder.encode(
        [q["reference_answer"] for q in qs], normalize_embeddings=True, show_progress_bar=False)
    rows = []
    for i, (q, ref_emb) in enumerate(zip(qs, ref_embs), 1):
        t0 = time.time()
        response = generate_fn(q["question"])
        latency = round(time.time() - t0, 2)
        rows.append({
            "id": q["id"], "category": q["category"], "difficulty": q["difficulty"],
            "question": q["question"], "response": response,
            "term_coverage": term_coverage_score(response, q["key_terms"]),
            "semantic_relevance": semantic_relevance_score(response, ref_emb),
            "latency_sec": latency,
        })
        if verbose and i % 10 == 0:
            print(f"  [{i:02d}/{len(qs)}] evaluated...")
    df = pd.DataFrame(rows)
    df["model"] = model_label
    return df

def summarize_medilearn(df):
    """Returns (by-difficulty summary, overall summary)."""
    by_diff = df.groupby("difficulty")[["term_coverage", "semantic_relevance", "latency_sec"]].mean().round(3)
    by_diff = by_diff.reindex(["L1", "L2", "L3"])
    overall = df[["term_coverage", "semantic_relevance", "latency_sec"]].mean().round(3)
    return by_diff, overall

print("✅ Evaluation harness ready")

def display_sample_comparison(baseline_df, finetuned_df, step: int = 5):
    """60 rows is too many to render inline — show an evenly-spaced sample
    (every `step`-th question) spanning all 6 categories instead. Full data
    still lives in baseline_df / finetuned_df and the saved JSON files."""
    idx = list(range(0, len(baseline_df), step))
    rows = []
    for i in idx:
        b, f = baseline_df.iloc[i], finetuned_df.iloc[i]
        d = round(f["term_coverage"] - b["term_coverage"], 3)
        col = "#27ae60" if d > 0 else ("#e74c3c" if d < 0 else "#7f8c8d")
        rows.append(f"""
        <tr>
          <td style='font-weight:bold;color:#2c3e50;vertical-align:top;padding:8px'>{b['id']}</td>
          <td style='vertical-align:top;padding:8px;font-size:11px;color:#7f8c8d'>{b['category']} / {b['difficulty']}</td>
          <td style='vertical-align:top;padding:8px;font-size:12px'>{b['question']}</td>
          <td style='vertical-align:top;padding:8px;font-size:12px;background:#fff5f5'>{b['response'][:200]}</td>
          <td style='text-align:center;vertical-align:top;padding:8px;font-weight:bold'>{b['term_coverage']}</td>
          <td style='vertical-align:top;padding:8px;font-size:12px;background:#f5fff5'>{f['response'][:200]}</td>
          <td style='text-align:center;vertical-align:top;padding:8px;font-weight:bold'>{f['term_coverage']}</td>
          <td style='text-align:center;vertical-align:top;padding:8px;font-weight:bold;color:{col}'>{d:+.3f}</td>
        </tr>""")
    html = f"""
    <style>
      .cmp{{border-collapse:collapse;width:100%;font-family:sans-serif;font-size:13px}}
      .cmp th{{background:#2c3e50;color:white;padding:10px;text-align:left}}
      .cmp tr:nth-child(even){{background:#f8f9fa}}
      .cmp td{{border:1px solid #dee2e6;vertical-align:top}}
    </style>
    <p style="font-size:12px;color:#7f8c8d">Showing {len(idx)} of {len(baseline_df)} questions (every {step}th, spanning all categories).</p>
    <table class='cmp'>
      <tr><th>ID</th><th>Category / Tier</th><th>Question</th>
          <th>🔵 Baseline Response</th><th>Term cov.</th>
          <th>🟢 Fine-Tuned Response</th><th>Term cov.</th><th>Δ</th></tr>
      {{''.join(rows)}}
    </table>"""
    display(HTML(html))

print("✅ Helper functions ready")

pyenv: version `3.13' is not installed (set by /Users/sudhanshusingh/Projects/DHS-SLM-Workshop/.python-version)
pyenv: pip: command not found

The `pip' command exists in these Python versions:
  3.12.13
  3.14.4
  miniforge3-26.3.2-0

Note: See 'pyenv help global' for tips on allowing multiple
      Python versions to be found at the same time.
Loading embedding model for semantic-relevance scoring (BAAI/bge-small-en-v1.5)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19629.51it/s]


✅ Evaluation harness ready
✅ Helper functions ready


---
## 🔵 Section 6 — Baseline Evaluation (Original SmolLM-135M-Instruct)

We load the **unmodified** base model — no medical fine-tuning at all — and run the full shared 60-question MediLearn set against it.

### What to expect
- **Verbose, markdown-heavy** answers that drift off-topic
- Medical terms misunderstood or confused with unrelated concepts  
  *(the model card example: "Bowman's space" confused with spacecraft)*
- This is our **control condition** — everything else is measured against it


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading: {BASE_MODEL_ID}  (~270 MB fp32 / ~70 MB in 4-bit)\n")
baseline_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=BNB_CONFIG if DEVICE == "cuda" else None,
    torch_dtype=COMPUTE_DTYPE if DEVICE in ["cuda", "mps"] else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    baseline_model = baseline_model.to(DEVICE)
baseline_model.eval()

n_params = sum(p.numel() for p in baseline_model.parameters())
print(f"✅ Loaded  |  Params: {n_params/1e6:.1f}M  |  Device: {next(baseline_model.parameters()).device}")
if DEVICE == "cuda":
    print(f"💾  4-bit NF4 footprint: {torch.cuda.memory_allocated()/1e9:.2f} GB")


Loading: HuggingFaceTB/SmolLM-135M-Instruct  (~270 MB fp32 / ~70 MB in 4-bit)



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 11520.02it/s]


✅ Loaded  |  Params: 134.5M  |  Device: mps:0


In [9]:
def generate_baseline(question: str) -> str:
    return generate_response(baseline_model, baseline_tokenizer, question)

baseline_df = run_medilearn_eval(generate_baseline, "Baseline SmolLM-135M-Instruct")
by_diff_base, overall_base = summarize_medilearn(baseline_df)

# Save locally (kept alongside the adapter for the zip download) and to Drive (for cross-module tracking)
baseline_df.to_json(os.path.join(RESULTS_DIR, "baseline_results.json"), orient="records", indent=2)
baseline_df.to_json(os.path.join(DRIVE_RESULTS_DIR, "module2_baseline.json"), orient="records", indent=2)

print(f"\n📊 Baseline — overall:")
print(overall_base)
print(f"\n📊 Baseline — by difficulty tier:")
print(by_diff_base)
print("\n💾 Saved → eval_results/baseline_results.json and Drive/eval/results/module2_baseline.json")

  [10/60] evaluated...
  [20/60] evaluated...
  [30/60] evaluated...
  [40/60] evaluated...
  [50/60] evaluated...
  [60/60] evaluated...

📊 Baseline — overall:
term_coverage         0.257
semantic_relevance    0.818
latency_sec           1.501
dtype: float64

📊 Baseline — by difficulty tier:
            term_coverage  semantic_relevance  latency_sec
difficulty                                                
L1                  0.472               0.851        1.492
L2                  0.185               0.801        1.521
L3                  0.139               0.807        1.482

💾 Saved → eval_results/baseline_results.json and Drive/eval/results/module2_baseline.json


In [10]:
# Free GPU memory before fine-tuning
del baseline_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()
    torch.cuda.empty_cache()
    print(f"🧹 GPU freed — {torch.cuda.memory_allocated()/1e9:.2f} GB in use now")


🧹 GPU freed — 0.00 GB in use now


---
## 🟡 Section 7 — Continued Fine-Tuning (Epoch 11)

We check the shared Drive folder for an already fine-tuned model first. If one exists there from a previous run, we continue training **that** model (a fresh adapter on top of it); otherwise we fall back to the published 10-epoch PEFT adapter, loaded onto a **4-bit NF4-quantized** base model. Either way we run **one more epoch** with identical settings using HuggingFace `Trainer` directly (more stable across library versions than `SFTTrainer` for continued adapter training).

### Pipeline
```
Drive model (if it exists)  OR  SmolLM-135M-Instruct  (frozen base, 4-bit NF4)
        +
fresh LoRA adapter (if resuming)  OR  mohres/...10epochs adapter (bf16/fp16)
        ↓
  HF Trainer — 1 epoch on medical_meadow_medical_flashcards
        ↓
  ./smollm_epoch11/   (updated adapter saved here)
```


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

# ── Resume-from-Drive check ───────────────────────────────────────────────────
# MERGED_MODEL_DIR (defined in Section 1d) is the SAME Drive path Section 7.1
# saves the final merged model to, and the SAME path Modules 3, 4, and 6 read
# from. If a previously fine-tuned model already exists there, we continue
# training from IT instead of re-downloading the original published 10-epoch
# adapter — so re-running this notebook accumulates epochs (11 → 12 → 13...)
# instead of always restarting from the same fixed 10-epoch checkpoint.
resume_from_drive = os.path.exists(os.path.join(MERGED_MODEL_DIR, "config.json"))

print(f"Loading base model (4-bit NF4): "
      f"{MERGED_MODEL_DIR if resume_from_drive else BASE_MODEL_ID}")

_tokenizer_source = MERGED_MODEL_DIR if resume_from_drive else BASE_MODEL_ID
ft_tokenizer = AutoTokenizer.from_pretrained(_tokenizer_source)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "right"

base = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_DIR if resume_from_drive else BASE_MODEL_ID,
    quantization_config=BNB_CONFIG if DEVICE == "cuda" else None,
    torch_dtype=COMPUTE_DTYPE if DEVICE in ["cuda", "mps"] else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    base = base.to(DEVICE)

if DEVICE == "cuda":
    # Casts norms to fp32, enables gradient checkpointing, and preps the
    # 4-bit base for adapter training (QLoRA-style continued training).
    base = prepare_model_for_kbit_training(base)

if resume_from_drive:
    # A model already sitting at MERGED_MODEL_DIR has had all of its previous
    # LoRA adapters merged permanently into its weights (Section 7.1 always
    # merges before saving) — there is no separate adapter left to reload.
    # So we attach a FRESH LoRA adapter on top of these already-fine-tuned
    # weights, using the same rank/alpha/target_modules as every prior run.
    print(f"✅ Found an existing fine-tuned model at: {MERGED_MODEL_DIR}")
    print("   Attaching a new LoRA adapter on top of it (continuing training).")
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODS,
        bias="none",
        task_type="CAUSAL_LM",
    )
    ft_model = get_peft_model(base, lora_config)
else:
    print(f"ℹ️  No existing model found at {MERGED_MODEL_DIR}")
    print(f"   Loading the published adapter from Hugging Face: {FINETUNED_ID}")
    ft_model = PeftModel.from_pretrained(base, FINETUNED_ID, is_trainable=True)

ft_model.print_trainable_parameters()


Loading base model (4-bit NF4): HuggingFaceTB/SmolLM-135M-Instruct


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 12027.95it/s]


ℹ️  No existing model found at /Users/sudhanshusingh/Projects/DHS-SLM-Workshop/SLM_Workshop/medical_slm_finetuned
   Loading the published adapter from Hugging Face: mohres/SmolLM-135M-Instruct-medical_meadow_medical_flashcards-10epochs
trainable params: 921,600 || all params: 135,436,608 || trainable%: 0.6805


In [12]:
# ── Load & format dataset ─────────────────────────────────────────────────────
print(f"Loading dataset: {DATASET_ID}")
raw = load_dataset(DATASET_ID)
split_name = list(raw.keys())[0]
cols = raw[split_name].column_names
print(f"Columns: {cols}")

def format_example(ex):
    instruction = ex.get("input",  ex.get("instruction", ""))
    response    = ex.get("output", ex.get("response", ""))
    text = ft_tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction},
         {"role": "assistant", "content": response}],
        tokenize=False, add_generation_prompt=False,
    )
    tok = ft_tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN, padding="max_length")
    tok["labels"] = tok["input_ids"].copy()
    return tok

remove_cols = cols
formatted = raw[split_name].map(format_example, remove_columns=remove_cols, batched=False)
formatted.set_format("torch")

split = formatted.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(f"\n✅ Dataset ready  |  train: {len(train_ds):,}  |  eval: {len(eval_ds):,}")
print(f"\nExample keys: {list(train_ds[0].keys())}")


Loading dataset: medalpaca/medical_meadow_medical_flashcards


Generating train split: 100%|██████████| 33955/33955 [00:00<00:00, 201398.57 examples/s]


Columns: ['input', 'output', 'instruction']


Map: 100%|██████████| 33955/33955 [00:05<00:00, 6297.08 examples/s]


✅ Dataset ready  |  train: 30,559  |  eval: 3,396

Example keys: ['input_ids', 'attention_mask', 'labels']


In [13]:
# ── Configure Trainer ─────────────────────────────────────────────────────────
training_args = TrainingArguments(**TRAIN_CONFIG)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=ft_tokenizer,
    mlm=False,   # causal LM, not masked LM
)

trainer = Trainer(
    model         = ft_model,
    args          = training_args,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    data_collator = data_collator,
)

print("✅ Trainer configured")
print(f"   Steps per epoch : {len(train_ds) // (TRAIN_CONFIG['per_device_train_batch_size'] * TRAIN_CONFIG['gradient_accumulation_steps'])}")
print(f"   Logging every   : {TRAIN_CONFIG['logging_steps']} steps")
print("\n🚀 Starting training (epoch 11)… ~15–40 min on T4")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Trainer configured
   Steps per epoch : 954
   Logging every   : 10 steps

🚀 Starting training (epoch 11)… ~15–40 min on T4


In [14]:
train_result = trainer.train()

print("\n✅ Training complete!")
print(f"   Loss    : {train_result.training_loss:.4f}")
print(f"   Steps   : {train_result.global_step}")
print(f"   Runtime : {train_result.metrics.get('train_runtime', 0):.0f}s  "
      f"({train_result.metrics.get('train_runtime', 0)/60:.1f} min)")

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Save updated adapter
ft_model.save_pretrained(OUTPUT_DIR)
ft_tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Updated model saved → {OUTPUT_DIR}/")
print(f"   Files: {os.listdir(OUTPUT_DIR)}")

del trainer
if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()
    torch.cuda.empty_cache()


---
## 🟢 Section 8 — Post-Training Evaluation (Epoch 11 Model)

Same shared 60-question set, same inference settings — only the model weights changed.

> **Run this evaluation before the merge step below.** `merge_and_unload()` (Section 8.1) folds the
> adapter into the base weights and unwraps the PEFT container in place — `ft_model` is not reliable
> for inference afterward, so we score it here first while it's still the live, trainable adapter model.

### What may change
| Dimension | Expected direction |
|-----------|-------------------|
| Response style | Stays concise / flashcard-like |
| Medical relevance | Maintained or improved |
| Specific factual accuracy | May shift ± per question |
| Verbosity | Should remain low |


In [ ]:
ft_model.eval()

def generate_finetuned(question: str) -> str:
    return generate_response(ft_model, ft_tokenizer, question)

finetuned_df = run_medilearn_eval(generate_finetuned, "Fine-Tuned SmolLM-135M (Epoch 11)")
by_diff_ft, overall_ft = summarize_medilearn(finetuned_df)

finetuned_df.to_json(os.path.join(RESULTS_DIR, "finetuned_results.json"), orient="records", indent=2)
finetuned_df.to_json(os.path.join(DRIVE_RESULTS_DIR, "module2_finetuned.json"), orient="records", indent=2)

print(f"\n📊 Fine-Tuned — overall:")
print(overall_ft)
print(f"\n📊 Fine-Tuned — by difficulty tier:")
print(by_diff_ft)
print("\n💾 Saved → eval_results/finetuned_results.json and Drive/eval/results/module2_finetuned.json")

---
## 🔗 Section 8.1 — Merge Adapter & Save to Drive *(critical hand-off step)*

The adapter saved earlier (`./smollm_epoch11/`) only exists in this Colab session's local disk — it
disappears when the runtime recycles, and it's just the LoRA delta, not a model Modules 3/4/6 can
load directly.

This step folds the adapter into the base model's weights (`merge_and_unload()`) and saves the
**merged, standalone model** to `MERGED_MODEL_DIR` on Google Drive. This is the exact path Module 3,
Module 4, and Module 6 expect — without this cell, those notebooks silently fall back to an
un-fine-tuned base model with no warning, which breaks every comparison downstream.

*(This runs after the evaluation above, not before — `merge_and_unload()` unwraps the PEFT
container in place, so `ft_model` shouldn't be used for further inference once this cell runs.)*

In [ ]:
print(f"Merging LoRA adapter into base weights...")
merged_model = ft_model.merge_and_unload()

os.makedirs(MERGED_MODEL_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_MODEL_DIR)
ft_tokenizer.save_pretrained(MERGED_MODEL_DIR)

saved_files = os.listdir(MERGED_MODEL_DIR)
print(f"✅ Merged model saved → {MERGED_MODEL_DIR}")
print(f"   Files: {saved_files}")
assert "config.json" in saved_files, "Merge/save did not produce a loadable model — check the cell above for errors."
print("\n➡️  Modules 3, 4, and 6 will load the fine-tuned MediLearn model from exactly this path.")

del ft_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()
    torch.cuda.empty_cache()

---
## 📊 Section 9 — Comparison & Analysis

### 9.1 Summary Statistics


In [ ]:
delta_overall = (overall_ft - overall_base).round(3)

summary_df = pd.DataFrame({
    "Metric":        ["Term coverage (0-1)", "Semantic relevance (0-1)", "Latency (sec)"],
    "🔵 Baseline":   [overall_base["term_coverage"], overall_base["semantic_relevance"], overall_base["latency_sec"]],
    "🟢 Fine-Tuned": [overall_ft["term_coverage"], overall_ft["semantic_relevance"], overall_ft["latency_sec"]],
    "Δ Change":      [f"{delta_overall['term_coverage']:+.3f}", f"{delta_overall['semantic_relevance']:+.3f}", f"{delta_overall['latency_sec']:+.2f}"],
}).set_index("Metric")

def colour_delta(v):
    if isinstance(v, str) and v.startswith("+"): return "color: #27ae60; font-weight: bold"
    if isinstance(v, str) and v.startswith("-"): return "color: #e74c3c; font-weight: bold"
    return ""

style_fn = summary_df.style.map if hasattr(summary_df.style, "map") else summary_df.style.applymap
display(style_fn(colour_delta, subset=["Δ Change"]))

print("\nBy difficulty tier (term coverage):")
tier_compare = pd.DataFrame({"Baseline": by_diff_base["term_coverage"], "Fine-Tuned": by_diff_ft["term_coverage"]})
tier_compare["Δ"] = (tier_compare["Fine-Tuned"] - tier_compare["Baseline"]).round(3)
display(tier_compare)

### 9.2 Score Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("SmolLM-135M — Baseline vs Fine-Tuned (Epoch 11), by difficulty tier", fontsize=13, fontweight="bold")

tiers = ["L1", "L2", "L3"]
tier_labels = ["L1\nBasic recall", "L2\nClinical scenario", "L3\nAdvanced reasoning"]
x, w = np.arange(len(tiers)), 0.35

ax = axes[0]
ax.bar(x - w/2, by_diff_base["term_coverage"], w, label="🔵 Baseline", color="#3498db", alpha=0.85)
ax.bar(x + w/2, by_diff_ft["term_coverage"], w, label="🟢 Fine-Tuned", color="#2ecc71", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(tier_labels, fontsize=9)
ax.set_ylabel("Term coverage (0-1)"); ax.set_ylim(0, 1)
ax.set_title("Accuracy by difficulty tier"); ax.legend(fontsize=8)

ax2 = axes[1]
deltas_by_tier = (by_diff_ft["term_coverage"] - by_diff_base["term_coverage"]).round(3)
colours = ["#27ae60" if d >= 0 else "#e74c3c" for d in deltas_by_tier]
bars = ax2.bar(tier_labels, deltas_by_tier, color=colours, alpha=0.85, edgecolor="white")
ax2.axhline(0, color="black", lw=0.8)
ax2.set_ylabel("Δ Term coverage (Fine-Tuned − Baseline)")
ax2.set_title("Improvement by difficulty tier")
ax2.tick_params(axis="x", labelsize=8)
for bar, d in zip(bars, deltas_by_tier):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (0.01 if d >= 0 else -0.04),
              f"{d:+.2f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "score_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved → eval_results/score_comparison.png")

### 9.3 Side-by-Side Response Table

In [ ]:
display_sample_comparison(baseline_df, finetuned_df, step=5)

### 9.4 Qualitative Analysis

#### 1. 🎯 Accuracy

| Model | Observation |
|-------|-------------|
| **Baseline** | Off-topic or medically wrong. No exposure to the clinical domain — drifts to general language patterns (e.g. "Bowman's space" → spacecraft analogy). |
| **Fine-Tuned (ep 11)** | Stays within the medical domain. Some specific facts still wrong (ep-10 model named erythromycin instead of primaquine), but the *type* of answer is always appropriate. |

> **Takeaway:** Fine-tuning improves domain relevance dramatically, even when factual precision isn't perfect.

---

#### 2. 📏 Completeness

| Model | Observation |
|-------|-------------|
| **Baseline** | Either too verbose (long markdown bullet lists) or evasive — talks around the question without answering it. |
| **Fine-Tuned** | Concise single-sentence answers that match the flashcard format. The model learns the *shape* of a correct answer from the training data. |

> **Takeaway:** Fine-tuning teaches the model *how* to answer, not just what vocabulary to use.

---

#### 3. 🎓 Relevance

| Model | Observation |
|-------|-------------|
| **Baseline** | Low — responses drift to general biology, chemistry, or completely unrelated concepts. |
| **Fine-Tuned** | High — every response addresses the right medical topic, even if the specific answer is imperfect. |

---

#### 4. 🖊️ Style / Tone

| Model | Observation |
|-------|-------------|
| **Baseline** | Markdown-heavy: headers, bold text, numbered lists. Generic assistant voice. |
| **Fine-Tuned** | Plain clinical prose, single sentence. Mirrors the flashcard dataset format. |

> **Takeaway:** Style adaptation is one of the clearest and most consistent wins from SFT.

---

#### 5. ⚠️ Regressions & Limitations

- **Overfitting risk** — With ~34k examples and 135M parameters, epoch 11 may reinforce dataset memorisation over generalisation
- **Factual errors persist** — Hallucinations require larger models or RLHF to reduce meaningfully
- **Domain narrowing** — The fine-tuned model may score worse on non-medical tasks
- **No clinical validation** — For research only; never use for medical decision-making


---
## 💾 Section 10 — Save Full Report & Download

The merged model is already saved to Drive (Section 7.1) — this section saves the evaluation
artifacts and packages the adapter for download.


In [ ]:
combined = {
    "run_timestamp":    datetime.now().isoformat(),
    "device":          DEVICE,
    "base_model":      BASE_MODEL_ID,
    "finetuned_model": FINETUNED_ID,
    "merged_model_path": MERGED_MODEL_DIR,
    "extra_epochs":    1,
    "eval_set":        "medilearn_eval_60 (shared across Modules 1, 2, 3, 4, 6)",
    "library_versions": {
        "transformers": __import__("transformers").__version__,
        "peft":         __import__("peft").__version__,
        "trl":          __import__("trl").__version__,
    },
    "summary_overall": {
        "baseline":   overall_base.to_dict(),
        "finetuned":  overall_ft.to_dict(),
        "delta":      delta_overall.to_dict(),
    },
    "summary_by_difficulty": {
        "baseline":  by_diff_base.to_dict(orient="index"),
        "finetuned": by_diff_ft.to_dict(orient="index"),
    },
}
with open(os.path.join(RESULTS_DIR, "combined_results.json"), "w") as fp:
    json.dump(combined, fp, indent=2)
print("✅ combined_results.json saved")

print("\n📁 All output files:")
for fname in sorted(os.listdir(RESULTS_DIR)):
    sz = os.path.getsize(os.path.join(RESULTS_DIR, fname))
    print(f"   {fname:45s} {sz:>8,} bytes")

In [ ]:
# Download everything as a ZIP (Colab only)
import sys
import os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import files
    import zipfile

    zip_path = "/content/smollm_eval_results.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in os.listdir(RESULTS_DIR):
            zf.write(os.path.join(RESULTS_DIR, fname), fname)
        # also include the saved adapter
        for fname in os.listdir(OUTPUT_DIR):
            zf.write(os.path.join(OUTPUT_DIR, fname), os.path.join("smollm_epoch11", fname))

    files.download(zip_path)
    print("✅ Download started — check your browser's download folder")
else:
    print("✅ Local environment: files are already saved in the local workspace directory.")


---
## 🏁 Summary & Key Takeaways

### What we demonstrated

| Stage | Model | Characteristic |
|-------|-------|----------------|
| Baseline | SmolLM-135M (no medical training) | Generic, verbose, off-topic |
| 10-epoch FT | `mohres/.../10epochs` | Concise, medical style, some factual errors |
| 11-epoch FT | This notebook's output | Incremental refinement of epoch-10 behaviour |

### 🔑 Key takeaways for your audience

> **1. Fine-tuning teaches *how* to answer, not just what vocabulary to use.**  
> The clearest win is response style and domain focus — the model learns to sound like a medical flashcard even before getting the specific fact right.

> **2. LoRA makes fine-tuning accessible.**  
> Only ~0.15% of the model's parameters were trained. Epoch 11 ran in ~20 min on a free T4 GPU.

> **3. More epochs ≠ always better.**  
> After 10 epochs on a 135M model, gains are incremental. Overfitting risk increases with each additional pass over the same data.

> **4. Systematic evaluation is as important as training.**  
> Without a fixed evaluation set and a consistent scoring function, it's impossible to know whether fine-tuning helped or hurt — which is why this notebook uses the same 60-question set, the same metrics, and the same scoring code as every other module in the workshop.

> **5. The merged model is the real deliverable.**  
> Section 7.1 merged the LoRA adapter into the base model and saved it to Google Drive. That's what Modules 3, 4, and 6 actually load — the adapter zip below is a convenience copy, not the hand-off artifact.

---
*This notebook is for research and educational purposes only.  
Model outputs must not be used for clinical decision-making.*
